In [1]:
!pip install -q llama-index
!pip install -q llama-index-retrievers-bm25
!pip install tabulate

In [2]:
import sys
import time
import pickle
import logging
from pathlib import Path
from typing import List, Optional, Tuple, Union
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz
from llama_index.core import Document
# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents, select_random_documents
from src.utils.evaluation import (
    perform_analysis,
    perform_single_match_analysis,
    print_single_match_analysis,
    print_sample_results,
    calculate_prefix_success_at_k,
    perform_prefix_match_analysis,
    print_prefix_match_analysis
)
from contextlib import contextmanager

# ---------------------------
# Timing context manager
# ---------------------------
@contextmanager
def timer(name):
    omitted_names = {
        "Selecting a random document for sanity check",
        "Encoding query",
        "Searching in Milvus",
        "Processing search results"
    }
    start = time.time()
    yield
    end = time.time()
    if name not in omitted_names and not name.startswith("Retrieval for document"):
        print(f"[Timing] {name}: {end - start:.4f} seconds")

logging.basicConfig(level=logging.INFO)

class SparseBM25Vectorizer:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.vectorizer = CountVectorizer(binary=False, lowercase=True, token_pattern=r'\b\w+\b')
        self.doc_len = None
        self.avgdl = None
        self.idf = None
        self.doc_freq = None

    def fit(self, documents: List[Union[Document, str]]):
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        X = self.vectorizer.fit_transform(texts)
        self.doc_len = X.sum(axis=1).A1
        self.avgdl = self.doc_len.mean()
        n_docs = X.shape[0]
        self.doc_freq = X.sum(axis=0).A1
        self.idf = np.log((n_docs - self.doc_freq + 0.5) / (self.doc_freq + 0.5) + 1.0)
        return self

    def transform(self, documents: Union[List[Union[Document, str]], str]):
        if isinstance(documents, str):
            documents = [documents]
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        X = self.vectorizer.transform(texts)
        return self._bm25_sparse(X)

    def _bm25_sparse(self, X):
        rows, cols = X.nonzero()
        data = X.data
        
        doc_len = self.doc_len[rows]
        idf = self.idf[cols]
        
        numerator = data * (self.k1 + 1)
        denominator = data + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
        
        scores = idf * numerator / denominator
        return csr_matrix((scores, (rows, cols)), shape=X.shape)

class SparseBM25Retriever:
    def __init__(self, documents, vectorizer, embeddings):
        self.documents = documents
        self.vectorizer = vectorizer
        self.embeddings = embeddings

    def retrieve(self, query: str, top_k=10):
        """
        Efficient retrieval of top-K relevant documents using sparse BM25.
        """
        query_embedding = self.vectorizer.transform([query])  # Ensure it's a sparse matrix
        
        # Compute sparse cosine similarity (dot product is enough in BM25 space)
        scores = query_embedding @ self.embeddings.T  # Sparse matrix multiplication
        scores = np.array(scores.toarray()).flatten()  # Convert to dense array
        
        # Get top-k indices efficiently using argpartition (O(N log K) instead of O(N log N))
        top_k_idx = np.argpartition(scores, -top_k)[-top_k:]  # Get top-K without sorting everything
        top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]  # Sort top-K results

        # Retrieve top results
        results = [
            {
                "item_key": self.documents[idx].metadata.get("item_key", f"Doc-{idx}"),
                "text": self.documents[idx].text,
                "score": scores[idx]
            }
            for idx in top_k_idx
        ]

        return results

def main(index_path_texto, index_path_resumen, file_identifier, batch_size=500, top_k=10):
    start_total = time.time()

    # Load persisted documents
    with open(index_path_texto / "documents.pkl", "rb") as f:
        documents_texto = pickle.load(f)
    with open(index_path_resumen / "documents.pkl", "rb") as f:
        documents_resumen = pickle.load(f)

    # Load persisted vectorizers and embeddings
    with open(index_path_texto / "vectorizer.pkl", "rb") as f:
        vectorizer_texto = pickle.load(f)
    embeddings_texto = load_npz(index_path_texto / "embeddings.npz")

    with open(index_path_resumen / "vectorizer.pkl", "rb") as f:
        vectorizer_resumen = pickle.load(f)
    # (We won’t use vectorizer_resumen here since we’re reusing vectorizer_texto for retrieval.)

    # Build a mapping from texto item keys to their indices for fast lookup
    texto_mapping = {doc.metadata.get("item_key"): idx for idx, doc in enumerate(documents_texto)}

    # Select a random sample from the resumen documents
    sample_size = min(len(documents_resumen), 16590)
    random_resumen = select_random_documents(documents_resumen, sample_size)
    resumen_keys, resumen_texts = zip(*random_resumen)  # unzip into separate lists

    all_results = []
    total_batches = (sample_size + batch_size - 1) // batch_size
    start_batches = time.time()

    # Batch processing: transform and compute scores in batches
    for batch_num, start_idx in enumerate(range(0, sample_size, batch_size), 1):
        end_idx = min(start_idx + batch_size, sample_size)
        batch_keys = resumen_keys[start_idx:end_idx]
        batch_texts = resumen_texts[start_idx:end_idx]

        # Batch-transform resumen texts using vectorizer_texto
        batch_query_embeddings = vectorizer_texto.transform(batch_texts)
        # Compute similarity scores in one sparse matrix multiplication
        # scores_batch: shape (batch_size, number of texto documents)
        scores_batch = (batch_query_embeddings @ embeddings_texto.T).toarray()

        # Process each query in the batch
        for i, score_row in enumerate(scores_batch):
            overall_idx = start_idx + i
            item_key = batch_keys[i]
            resumen_text = batch_texts[i]

            # Extract top_k indices and sort them by score descending
            topk_idx_unsorted = np.argpartition(score_row, -top_k)[-top_k:]
            topk_idx = topk_idx_unsorted[np.argsort(score_row[topk_idx_unsorted])[::-1]]

            # Build the list of top_k results
            texto_results = [{
                "item_key": documents_texto[idx].metadata.get("item_key", f"Doc-{idx}"),
                "text": documents_texto[idx].text,
                "score": score_row[idx]
            } for idx in topk_idx]

            # Get target score from matching texto document (if exists)
            texto_index = texto_mapping.get(item_key, None)
            if texto_index is not None:
                target_score = score_row[texto_index]
                target_text = documents_texto[texto_index].text
            else:
                target_score = 0.0
                target_text = "No matching texto found"

            # Append results for this resumen document
            all_results.append({
                'resumen_item_key': item_key,
                'resumen_text': resumen_text,
                'target_text': target_text,
                'target_score': target_score,
                'texto_results': texto_results
            })

        # Logging progress for each batch
        elapsed_batches = time.time() - start_batches
        batches_remaining = total_batches - batch_num
        avg_batch_time = elapsed_batches / batch_num
        estimated_remaining_sec = batches_remaining * avg_batch_time
        est_min, est_sec = divmod(estimated_remaining_sec, 60)
        logging.info(
            f"Batch {batch_num}/{total_batches} processed. "
            f"Estimated time to completion: {int(est_min)}m {int(est_sec)}s"
        )

    # Perform overall analysis
    analysis_results = perform_analysis(all_results)
    print("\nAnalysis Results:")
    for metric, value in analysis_results.items():
        print(f"{metric}: {value:.4f}")

    # Perform single-match analysis and print results
    metrics = perform_single_match_analysis(all_results, top_k=top_k)
    print_single_match_analysis(metrics, top_k=top_k)

    # Perform prefix-matching analysis
    with timer("Performing prefix-matching analysis"):
        prefix_length = 6  # Match first 6 characters of item_key
        prefix_match_metrics = perform_prefix_match_analysis(
            all_results, 
            top_k=10, 
            prefix_length=prefix_length
        )
        print_prefix_match_analysis(
            prefix_match_metrics, 
            top_k=10, 
            prefix_length=prefix_length
        )

    # Print some sample retrieval results
    print_sample_results(all_results)

    total_elapsed = time.time() - start_total
    tot_min, tot_sec = divmod(total_elapsed, 60)
    print(f"\nTotal execution time: {int(tot_min)}m {int(tot_sec)}s")


In [3]:
index_path_texto = Path(f'/work/sandbox/bm25/index/k10.6b0.0_texto')
index_path_resumen = Path(f'/work/sandbox/bm25/index/k10.6b0.0_resumen')
main(index_path_texto, index_path_resumen, 'OEB')

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.8841
Precision: 0.0884
F1 Score: 0.1607
Mean Reciprocal Rank (MRR): 0.5671
Mean Average Precision (MAP): 0.5671

Single-Match Analysis Results:
Success@10             : 88.41%
MRR: 0.5671
Mean Position: 2.43
Median Position: 2.0
Perfect Matches: 38.83%

Position Distribution:
------------------------------
Position_1: 38.83%
Position_2: 23.75%
Position_3: 7.36%
Position_4: 8.28%
Position_5: 1.78%
Position_6: 1.64%
Position_7: 2.54%
Position_8: 2.35%
Position_9: 0.92%
Position_10: 0.96%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.13%
Prefix_MRR: 0.9898
Prefix_Mean_Position: 1.01
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 98.94%

Prefix Position Distribution:
----------------------------------------
Position 1    : 98.94%
Position 2    : 0.05%
Position 3    : 0.01%
Position 4    : 0.00%
Position 5    : 0.00%
Position 6    : 0.00%
Position 7    : 0.13%
Position 8    : 0.00%
Position 9    : 0.00%
Position 

In [10]:
index_path_texto = Path(f'/work/sandbox/bm25/index/k11.5b0.75_texto')
index_path_resumen = Path(f'/work/sandbox/bm25/index/k11.5b0.75_resumen')
main(index_path_texto, index_path_resumen, 'OEB')

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 31s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9193
Precision: 0.0919
F1 Score: 0.1671
Mean Reciprocal Rank (MRR): 0.8506
Mean Average Precision (MAP): 0.8506

Single-Match Analysis Results:
Success@10             : 91.93%
MRR: 0.8506
Mean Position: 1.34
Median Position: 1.0
Perfect Matches: 82.18%

Position Distribution:
------------------------------
Position_1: 82.18%
Position_2: 2.20%
Position_3: 2.33%
Position_4: 2.16%
Position_5: 0.90%
Position_6: 0.46%
Position_7: 0.72%
Position_8: 0.35%
Position_9: 0.48%
Position_10: 0.15%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 95.26%
Prefix_MRR: 0.9257
Prefix_Mean_Position: 1.15
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 91.52%

Prefix Position Distribution:
----------------------------------------
Position 1    : 91.52%
Position 2    : 1.06%
Position 3    : 0.40%
Position 4    : 0.25%
Position 5    : 0.72%
Position 6    : 0.31%
Position 7    : 0.41%
Position 8    : 0.13%
Position 9    : 0.40%
Position 1

In [12]:
k1= [0.2, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50] #k1
b = [0.3, 0.35, 0.40, 0.45, 0.5] #b
for k_item in k1:
    for b_item in b:
        index_path_texto = Path(f'/work/sandbox/bm25/index/k1' + str(k_item) + 
                                'b' + str(b_item) + '_texto')
        index_path_resumen = Path(f'/work/sandbox/bm25/index/k1' + str(k_item) + 
                                'b' + str(b_item) + '_resumen')
        
        print('-'*50)
        print('k1 = ' + str(k_item) + ' | b = ' + str(b_item))
        main(index_path_texto, index_path_resumen, 'OEB')

--------------------------------------------------
k1 = 0.2 | b = 0.3


INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 31s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9775
Precision: 0.0977
F1 Score: 0.1777
Mean Reciprocal Rank (MRR): 0.9092
Mean Average Precision (MAP): 0.9092

Single-Match Analysis Results:
Success@10             : 97.75%
MRR: 0.9092
Mean Position: 1.33
Median Position: 1.0
Perfect Matches: 87.93%

Position Distribution:
------------------------------
Position_1: 87.93%
Position_2: 3.07%
Position_3: 1.42%
Position_4: 1.30%
Position_5: 1.50%
Position_6: 1.03%
Position_7: 0.39%
Position_8: 0.31%
Position_9: 0.51%
Position_10: 0.27%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.26%
Prefix_MRR: 0.9805
Prefix_Mean_Position: 1.07
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 97.68%

Prefix Position Distribution:
----------------------------------------
Position 1    : 97.68%
Position 2    : 0.14%
Position 3    : 0.39%
Position 4    : 0.15%
Position 5    : 0.39%
Position 6    : 0.03%
Position 7    : 0.10%
Position 8    : 0.00%
Position 9    : 0.35%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 31s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9763
Precision: 0.0976
F1 Score: 0.1775
Mean Reciprocal Rank (MRR): 0.9059
Mean Average Precision (MAP): 0.9059

Single-Match Analysis Results:
Success@10             : 97.63%
MRR: 0.9059
Mean Position: 1.35
Median Position: 1.0
Perfect Matches: 87.60%

Position Distribution:
------------------------------
Position_1: 87.60%
Position_2: 3.03%
Position_3: 1.42%
Position_4: 1.32%
Position_5: 1.36%
Position_6: 1.07%
Position_7: 0.52%
Position_8: 0.38%
Position_9: 0.66%
Position_10: 0.28%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.32%
Prefix_MRR: 0.9792
Prefix_Mean_Position: 1.09
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 97.51%

Prefix Position Distribution:
----------------------------------------
Position 1    : 97.51%
Position 2    : 0.17%
Position 3    : 0.37%
Position 4    : 0.14%
Position 5    : 0.31%
Position 6    : 0.00%
Position 7    : 0.28%
Position 8    : 0.08%
Position 9    : 0.41%
Position 1

...


Analysis Results:
Recall: 0.9625
Precision: 0.0963
F1 Score: 0.1750
Mean Reciprocal Rank (MRR): 0.9072
Mean Average Precision (MAP): 0.9072

Single-Match Analysis Results:
Success@10             : 96.25%
MRR: 0.9072
Mean Position: 1.29
Median Position: 1.0
Perfect Matches: 88.71%

Position Distribution:
------------------------------
Position_1: 88.71%
Position_2: 1.38%
Position_3: 1.31%
Position_4: 1.15%
Position_5: 1.33%
Position_6: 0.98%
Position_7: 0.19%
Position_8: 0.28%
Position_9: 0.75%
Position_10: 0.17%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 98.59%
Prefix_MRR: 0.9758
Prefix_Mean_Position: 1.06
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 97.30%

Prefix Position Distribution:
----------------------------------------
Position 1    : 97.30%
Position 2    : 0.12%
Position 3    : 0.24%
Position 4    : 0.11%
Position 5    : 0.23%
Position 6    : 0.00%
Position 7    : 0.16%
Position 8    : 0.05%
Position 9    : 0.36%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 34s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 31s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9567
Precision: 0.0957
F1 Score: 0.1739
Mean Reciprocal Rank (MRR): 0.9039
Mean Average Precision (MAP): 0.9039

Single-Match Analysis Results:
Success@10             : 95.67%
MRR: 0.9039
Mean Position: 1.28
Median Position: 1.0
Perfect Matches: 88.45%

Position Distribution:
------------------------------
Position_1: 88.45%
Position_2: 1.39%
Position_3: 1.10%
Position_4: 1.18%
Position_5: 1.45%
Position_6: 0.84%
Position_7: 0.19%
Position_8: 0.24%
Position_9: 0.60%
Position_10: 0.23%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 98.31%
Prefix_MRR: 0.9737
Prefix_Mean_Position: 1.06
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 97.12%

Prefix Position Distribution:
----------------------------------------
Position 1    : 97.12%
Position 2    : 0.12%
Position 3    : 0.13%
Position 4    : 0.08%
Position 5    : 0.26%
Position 6    : 0.04%
Position 7    : 0.14%
Position 8    : 0.04%
Position 9    : 0.37%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9562
Precision: 0.0956
F1 Score: 0.1739
Mean Reciprocal Rank (MRR): 0.9052
Mean Average Precision (MAP): 0.9052

Single-Match Analysis Results:
Success@10             : 95.62%
MRR: 0.9052
Mean Position: 1.26
Median Position: 1.0
Perfect Matches: 88.63%

Position Distribution:
------------------------------
Position_1: 88.63%
Position_2: 1.24%
Position_3: 1.29%
Position_4: 1.48%
Position_5: 1.05%
Position_6: 0.66%
Position_7: 0.28%
Position_8: 0.25%
Position_9: 0.61%
Position_10: 0.13%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 98.24%
Prefix_MRR: 0.9749
Prefix_Mean_Position: 1.05
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 97.32%

Prefix Position Distribution:
----------------------------------------
Position 1    : 97.32%
Position 2    : 0.09%
Position 3    : 0.08%
Position 4    : 0.01%
Position 5    : 0.07%
Position 6    : 0.05%
Position 7    : 0.19%
Position 8    : 0.07%
Position 9    : 0.34%
Position 1

In [13]:
k1= [0.05, 0.10, 0.15]
b = [0.1, 0.15, 0.20, 0.25, 0.30]
for k_item in k1:
    for b_item in b:
        index_path_texto = Path(f'/work/sandbox/bm25/index/k1' + str(k_item) + 
                                'b' + str(b_item) + '_texto')
        index_path_resumen = Path(f'/work/sandbox/bm25/index/k1' + str(k_item) + 
                                'b' + str(b_item) + '_resumen')
        
        print('-'*50)
        print('k1 = ' + str(k_item) + ' | b = ' + str(b_item))
        main(index_path_texto, index_path_resumen, 'OEB')

--------------------------------------------------
k1 = 0.05 | b = 0.1


INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 31s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9752
Precision: 0.0975
F1 Score: 0.1773
Mean Reciprocal Rank (MRR): 0.9168
Mean Average Precision (MAP): 0.9168

Single-Match Analysis Results:
Success@10             : 97.52%
MRR: 0.9168
Mean Position: 1.25
Median Position: 1.0
Perfect Matches: 88.88%

Position Distribution:
------------------------------
Position_1: 88.88%
Position_2: 2.82%
Position_3: 1.80%
Position_4: 1.22%
Position_5: 1.06%
Position_6: 1.16%
Position_7: 0.16%
Position_8: 0.11%
Position_9: 0.17%
Position_10: 0.11%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.37%
Prefix_MRR: 0.9924
Prefix_Mean_Position: 1.01
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 99.19%

Prefix Position Distribution:
----------------------------------------
Position 1    : 99.19%
Position 2    : 0.04%
Position 3    : 0.02%
Position 4    : 0.04%
Position 5    : 0.02%
Position 6    : 0.00%
Position 7    : 0.03%
Position 8    : 0.03%
Position 9    : 0.00%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 31s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9787
Precision: 0.0979
F1 Score: 0.1779
Mean Reciprocal Rank (MRR): 0.9210
Mean Average Precision (MAP): 0.9210

Single-Match Analysis Results:
Success@10             : 97.87%
MRR: 0.9210
Mean Position: 1.26
Median Position: 1.0
Perfect Matches: 89.37%

Position Distribution:
------------------------------
Position_1: 89.37%
Position_2: 3.07%
Position_3: 1.21%
Position_4: 1.18%
Position_5: 1.07%
Position_6: 1.09%
Position_7: 0.25%
Position_8: 0.20%
Position_9: 0.28%
Position_10: 0.16%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.36%
Prefix_MRR: 0.9921
Prefix_Mean_Position: 1.01
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 99.16%

Prefix Position Distribution:
----------------------------------------
Position 1    : 99.16%
Position 2    : 0.05%
Position 3    : 0.01%
Position 4    : 0.04%
Position 5    : 0.01%
Position 6    : 0.00%
Position 7    : 0.06%
Position 8    : 0.03%
Position 9    : 0.00%
Position 1

...


Analysis Results:
Recall: 0.9775
Precision: 0.0977
F1 Score: 0.1777
Mean Reciprocal Rank (MRR): 0.9128
Mean Average Precision (MAP): 0.9128

Single-Match Analysis Results:
Success@10             : 97.75%
MRR: 0.9128
Mean Position: 1.30
Median Position: 1.0
Perfect Matches: 88.32%

Position Distribution:
------------------------------
Position_1: 88.32%
Position_2: 3.23%
Position_3: 1.31%
Position_4: 1.25%
Position_5: 1.35%
Position_6: 1.13%
Position_7: 0.23%
Position_8: 0.27%
Position_9: 0.35%
Position_10: 0.31%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.39%
Prefix_MRR: 0.9879
Prefix_Mean_Position: 1.03
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 98.57%

Prefix Position Distribution:
----------------------------------------
Position 1    : 98.57%
Position 2    : 0.11%
Position 3    : 0.28%
Position 4    : 0.08%
Position 5    : 0.20%
Position 6    : 0.00%
Position 7    : 0.05%
Position 8    : 0.01%
Position 9    : 0.05%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9793
Precision: 0.0979
F1 Score: 0.1781
Mean Reciprocal Rank (MRR): 0.9132
Mean Average Precision (MAP): 0.9132

Single-Match Analysis Results:
Success@10             : 97.93%
MRR: 0.9132
Mean Position: 1.32
Median Position: 1.0
Perfect Matches: 88.38%

Position Distribution:
------------------------------
Position_1: 88.38%
Position_2: 3.16%
Position_3: 1.30%
Position_4: 1.07%
Position_5: 1.54%
Position_6: 1.15%
Position_7: 0.34%
Position_8: 0.20%
Position_9: 0.45%
Position_10: 0.34%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.36%
Prefix_MRR: 0.9836
Prefix_Mean_Position: 1.05
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 97.98%

Prefix Position Distribution:
----------------------------------------
Position 1    : 97.98%
Position 2    : 0.25%
Position 3    : 0.33%
Position 4    : 0.07%
Position 5    : 0.43%
Position 6    : 0.00%
Position 7    : 0.11%
Position 8    : 0.02%
Position 9    : 0.10%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9797
Precision: 0.0980
F1 Score: 0.1781
Mean Reciprocal Rank (MRR): 0.9079
Mean Average Precision (MAP): 0.9079

Single-Match Analysis Results:
Success@10             : 97.97%
MRR: 0.9079
Mean Position: 1.35
Median Position: 1.0
Perfect Matches: 87.69%

Position Distribution:
------------------------------
Position_1: 87.69%
Position_2: 3.22%
Position_3: 1.42%
Position_4: 1.17%
Position_5: 1.72%
Position_6: 1.14%
Position_7: 0.39%
Position_8: 0.27%
Position_9: 0.58%
Position_10: 0.37%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.34%
Prefix_MRR: 0.9811
Prefix_Mean_Position: 1.07
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 97.70%

Prefix Position Distribution:
----------------------------------------
Position 1    : 97.70%
Position 2    : 0.21%
Position 3    : 0.34%
Position 4    : 0.15%
Position 5    : 0.46%
Position 6    : 0.00%
Position 7    : 0.16%
Position 8    : 0.05%
Position 9    : 0.23%
Position 1

In [19]:
rows = [0.05, 0.10, 0.15] #k1
columns = [0.1, 0.15, 0.20, 0.25, 0.30] #b

matrix_single = [
    [97.52, 97.87, 98.02, 98.17, 98.31],  
    [97.43, 97.72, 97.95, 98.13, 98.18],
    [97.04, 97.36, 97.75, 97.93, 97.97],
]

matrix_prefix = [
    [99.37, 99.36, 99.31, 99.46, 99.39],  
    [99.31, 99.36, 99.22, 99.36, 99.33],
    [99.38, 99.34, 99.39, 99.36, 99.34],
]


from tabulate import tabulate

# Prepare header (with one empty cell top-left)
headers = ["k1 \\ b"] + columns

# Combine each row with its corresponding k1 value
table_single = [[rows[i]] + matrix_single[i] for i in range(len(rows))]
table_prefix = [[rows[i]] + matrix_prefix[i] for i in range(len(rows))]

# Print the table
print('Single-match')
print(tabulate(table_single, headers=headers, floatfmt=".2f"))
print('-'*50)
print('Prefix-match')
print(tabulate(table_prefix, headers=headers, floatfmt=".2f"))

Single-match
  k1 \ b    0.1    0.15    0.2    0.25    0.3
--------  -----  ------  -----  ------  -----
    0.05  97.52   97.87  98.02   98.17  98.31
    0.10  97.43   97.72  97.95   98.13  98.18
    0.15  97.04   97.36  97.75   97.93  97.97
--------------------------------------------------
Prefix-match
  k1 \ b    0.1    0.15    0.2    0.25    0.3
--------  -----  ------  -----  ------  -----
    0.05  99.37   99.36  99.31   99.46  99.39
    0.10  99.31   99.36  99.22   99.36  99.33
    0.15  99.38   99.34  99.39   99.36  99.34


In [ ]:
rows = [0.0, 0.05, 0.1]  #k1
columns = [0.25, 0.3, 0.35] #b

matrix_single = [
    [, ,],  
    [, ,],
    [, ,],
]

matrix_prefix = [
    [, ,],  
    [, ,],
    [, ,],
]


from tabulate import tabulate

# Prepare header (with one empty cell top-left)
headers = ["k1 \\ b"] + columns

# Combine each row with its corresponding k1 value
table_single = [[rows[i]] + matrix_single[i] for i in range(len(rows))]
table_prefix = [[rows[i]] + matrix_prefix[i] for i in range(len(rows))]

# Print the table
print('Single-match')
print(tabulate(table_single, headers=headers, floatfmt=".2f"))
print('-'*50)
print('Prefix-match')
print(tabulate(table_prefix, headers=headers, floatfmt=".2f"))

In [22]:
k1= [0.0, 0.05, 0.1] 
b = [0.25, 0.3, 0.35]
for k_item in k1:
    for b_item in b:
        index_path_texto = Path(f'/work/sandbox/bm25/index/k1' + str(k_item) + 
                                'b' + str(b_item) + '_texto')
        index_path_resumen = Path(f'/work/sandbox/bm25/index/k1' + str(k_item) + 
                                'b' + str(b_item) + '_resumen')
        
        print('-'*50)
        print('k1 = ' + str(k_item) + ' | b = ' + str(b_item))
        main(index_path_texto, index_path_resumen, 'OEB')

--------------------------------------------------
k1 = 0.0 | b = 0.25


INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 33s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.8627
Precision: 0.0863
F1 Score: 0.1569
Mean Reciprocal Rank (MRR): 0.5517
Mean Average Precision (MAP): 0.5517

Single-Match Analysis Results:
Success@10             : 86.27%
MRR: 0.5517
Mean Position: 2.39
Median Position: 2.0
Perfect Matches: 37.31%

Position Distribution:
------------------------------
Position_1: 37.31%
Position_2: 23.71%
Position_3: 7.78%
Position_4: 7.95%
Position_5: 2.11%
Position_6: 2.04%
Position_7: 1.86%
Position_8: 1.69%
Position_9: 0.92%
Position_10: 0.92%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.34%
Prefix_MRR: 0.9923
Prefix_Mean_Position: 1.00
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 99.14%

Prefix Position Distribution:
----------------------------------------
Position 1    : 99.14%
Position 2    : 0.16%
Position 3    : 0.04%
Position 4    : 0.01%
Position 5    : 0.01%
Position 6    : 0.00%
Position 7    : 0.00%
Position 8    : 0.00%
Position 9    : 0.00%
Position 

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 31s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.8668
Precision: 0.0867
F1 Score: 0.1576
Mean Reciprocal Rank (MRR): 0.5536
Mean Average Precision (MAP): 0.5536

Single-Match Analysis Results:
Success@10             : 86.68%
MRR: 0.5536
Mean Position: 2.41
Median Position: 2.0
Perfect Matches: 37.58%

Position Distribution:
------------------------------
Position_1: 37.58%
Position_2: 23.14%
Position_3: 8.38%
Position_4: 7.89%
Position_5: 1.99%
Position_6: 2.07%
Position_7: 2.10%
Position_8: 1.80%
Position_9: 0.80%
Position_10: 0.92%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.32%
Prefix_MRR: 0.9921
Prefix_Mean_Position: 1.00
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 99.12%

Prefix Position Distribution:
----------------------------------------
Position 1    : 99.12%
Position 2    : 0.13%
Position 3    : 0.05%
Position 4    : 0.01%
Position 5    : 0.01%
Position 6    : 0.00%
Position 7    : 0.00%
Position 8    : 0.00%
Position 9    : 0.00%
Position 

...


Analysis Results:
Recall: 0.9821
Precision: 0.0982
F1 Score: 0.1786
Mean Reciprocal Rank (MRR): 0.9168
Mean Average Precision (MAP): 0.9168

Single-Match Analysis Results:
Success@10             : 98.21%
MRR: 0.9168
Mean Position: 1.30
Median Position: 1.0
Perfect Matches: 88.64%

Position Distribution:
------------------------------
Position_1: 88.64%
Position_2: 3.36%
Position_3: 1.29%
Position_4: 1.21%
Position_5: 1.43%
Position_6: 1.30%
Position_7: 0.31%
Position_8: 0.18%
Position_9: 0.28%
Position_10: 0.20%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.40%
Prefix_MRR: 0.9865
Prefix_Mean_Position: 1.03
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 98.33%

Prefix Position Distribution:
----------------------------------------
Position 1    : 98.33%
Position 2    : 0.21%
Position 3    : 0.32%
Position 4    : 0.11%
Position 5    : 0.27%
Position 6    : 0.01%
Position 7    : 0.10%
Position 8    : 0.03%
Position 9    : 0.01%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 31s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9821
Precision: 0.0982
F1 Score: 0.1786
Mean Reciprocal Rank (MRR): 0.9151
Mean Average Precision (MAP): 0.9151

Single-Match Analysis Results:
Success@10             : 98.21%
MRR: 0.9151
Mean Position: 1.31
Median Position: 1.0
Perfect Matches: 88.54%

Position Distribution:
------------------------------
Position_1: 88.54%
Position_2: 3.10%
Position_3: 1.45%
Position_4: 1.08%
Position_5: 1.53%
Position_6: 1.32%
Position_7: 0.34%
Position_8: 0.22%
Position_9: 0.37%
Position_10: 0.27%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.34%
Prefix_MRR: 0.9841
Prefix_Mean_Position: 1.05
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 98.06%

Prefix Position Distribution:
----------------------------------------
Position 1    : 98.06%
Position 2    : 0.24%
Position 3    : 0.34%
Position 4    : 0.07%
Position 5    : 0.34%
Position 6    : 0.02%
Position 7    : 0.11%
Position 8    : 0.03%
Position 9    : 0.11%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9794
Precision: 0.0979
F1 Score: 0.1781
Mean Reciprocal Rank (MRR): 0.9128
Mean Average Precision (MAP): 0.9128

Single-Match Analysis Results:
Success@10             : 97.94%
MRR: 0.9128
Mean Position: 1.31
Median Position: 1.0
Perfect Matches: 88.24%

Position Distribution:
------------------------------
Position_1: 88.24%
Position_2: 3.27%
Position_3: 1.40%
Position_4: 1.25%
Position_5: 1.46%
Position_6: 1.12%
Position_7: 0.35%
Position_8: 0.22%
Position_9: 0.40%
Position_10: 0.23%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.30%
Prefix_MRR: 0.9822
Prefix_Mean_Position: 1.06
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 97.85%

Prefix Position Distribution:
----------------------------------------
Position 1    : 97.85%
Position 2    : 0.20%
Position 3    : 0.34%
Position 4    : 0.14%
Position 5    : 0.34%
Position 6    : 0.01%
Position 7    : 0.11%
Position 8    : 0.04%
Position 9    : 0.25%
Position 1

In [23]:
rows = [0.0, 0.05, 0.1]  # k1
columns = [0.25, 0.30, 0.35] # b

matrix_single = [
    [86.27, 86.68, 86.15],  
    [98.20, 98.13, 98.21],  
    [98.21, 98.21, 97.94]
]

matrix_prefix = [
    [99.34, 99.32, 99.35],  
    [99.34, 99.32, 99.36],  
    [99.40, 99.34, 99.30]
]


from tabulate import tabulate

# Prepare header (with one empty cell top-left)
headers = ["k1 \\ b"] + columns

# Combine each row with its corresponding k1 value
table_single = [[rows[i]] + matrix_single[i] for i in range(len(rows))]
table_prefix = [[rows[i]] + matrix_prefix[i] for i in range(len(rows))]

# Print the table
print('Single-match')
print(tabulate(table_single, headers=headers, floatfmt=".2f"))
print('-'*50)
print('Prefix-match')
print(tabulate(table_prefix, headers=headers, floatfmt=".2f"))

Single-match
  k1 \ b    0.25    0.3    0.35
--------  ------  -----  ------
    0.00   86.27  86.68   86.15
    0.05   98.20  98.13   98.21
    0.10   98.21  98.21   97.94
--------------------------------------------------
Prefix-match
  k1 \ b    0.25    0.3    0.35
--------  ------  -----  ------
    0.00   99.34  99.32   99.35
    0.05   99.34  99.32   99.36
    0.10   99.40  99.34   99.30


### Single-match

| k1 \ b | 0.25  | 0.30  | 0.35  |
|--------|-------|-------|-------|
|  0.00  | 86.27 | 86.68 | 86.15 |
|  0.05  | 98.20 | 98.13 | 98.21 |
|  0.10  | 98.21 | 98.21 | 97.94 |

---

### Prefix-match

| k1 \ b | 0.25  | 0.30  | 0.35  |
|--------|-------|-------|-------|
|  0.00  | 99.34 | 99.32 | 99.35 |
|  0.05  | 99.34 | 99.32 | 99.36 |
|  0.10  | 99.40 | 99.34 | 99.30 |


In [17]:
k1= [0.05, 0.05, 0.1] 
b = [0.15, 0.20, 0.25]
for k_item in k1:
    for b_item in b:
        index_path_texto = Path(f'/work/sandbox/bm25/index/k1' + str(k_item) + 
                                'b' + str(b_item) + '_texto')
        index_path_resumen = Path(f'/work/sandbox/bm25/index/k1' + str(k_item) + 
                                'b' + str(b_item) + '_resumen')
        
        print('-'*50)
        print('k1 = ' + str(k_item) + ' | b = ' + str(b_item))
        main(index_path_texto, index_path_resumen, 'OEB')

--------------------------------------------------
k1 = 0.05 | b = 0.15


INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 31s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9781
Precision: 0.0978
F1 Score: 0.1778
Mean Reciprocal Rank (MRR): 0.9187
Mean Average Precision (MAP): 0.9187

Single-Match Analysis Results:
Success@10             : 97.81%
MRR: 0.9187
Mean Position: 1.26
Median Position: 1.0
Perfect Matches: 88.99%

Position Distribution:
------------------------------
Position_1: 88.99%
Position_2: 3.24%
Position_3: 1.28%
Position_4: 1.33%
Position_5: 1.13%
Position_6: 1.04%
Position_7: 0.22%
Position_8: 0.15%
Position_9: 0.26%
Position_10: 0.17%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.33%
Prefix_MRR: 0.9919
Prefix_Mean_Position: 1.01
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 99.11%

Prefix Position Distribution:
----------------------------------------
Position 1    : 99.11%
Position 2    : 0.09%
Position 3    : 0.02%
Position 4    : 0.04%
Position 5    : 0.03%
Position 6    : 0.00%
Position 7    : 0.03%
Position 8    : 0.01%
Position 9    : 0.00%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 31s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9816
Precision: 0.0982
F1 Score: 0.1785
Mean Reciprocal Rank (MRR): 0.9199
Mean Average Precision (MAP): 0.9199

Single-Match Analysis Results:
Success@10             : 98.16%
MRR: 0.9199
Mean Position: 1.27
Median Position: 1.0
Perfect Matches: 89.08%

Position Distribution:
------------------------------
Position_1: 89.08%
Position_2: 3.19%
Position_3: 1.37%
Position_4: 1.13%
Position_5: 1.33%
Position_6: 1.18%
Position_7: 0.35%
Position_8: 0.16%
Position_9: 0.21%
Position_10: 0.14%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.44%
Prefix_MRR: 0.9924
Prefix_Mean_Position: 1.01
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 99.14%

Prefix Position Distribution:
----------------------------------------
Position 1    : 99.14%
Position 2    : 0.13%
Position 3    : 0.05%
Position 4    : 0.04%
Position 5    : 0.04%
Position 6    : 0.00%
Position 7    : 0.02%
Position 8    : 0.02%
Position 9    : 0.00%
Position 1

...


Analysis Results:
Recall: 0.9788
Precision: 0.0979
F1 Score: 0.1780
Mean Reciprocal Rank (MRR): 0.9215
Mean Average Precision (MAP): 0.9215

Single-Match Analysis Results:
Success@10             : 97.88%
MRR: 0.9215
Mean Position: 1.25
Median Position: 1.0
Perfect Matches: 89.37%

Position Distribution:
------------------------------
Position_1: 89.37%
Position_2: 3.21%
Position_3: 1.22%
Position_4: 1.03%
Position_5: 1.19%
Position_6: 1.10%
Position_7: 0.13%
Position_8: 0.19%
Position_9: 0.23%
Position_10: 0.21%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.32%
Prefix_MRR: 0.9908
Prefix_Mean_Position: 1.01
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 98.98%

Prefix Position Distribution:
----------------------------------------
Position 1    : 98.98%
Position 2    : 0.13%
Position 3    : 0.07%
Position 4    : 0.02%
Position 5    : 0.01%
Position 6    : 0.01%
Position 7    : 0.04%
Position 8    : 0.03%
Position 9    : 0.02%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9812
Precision: 0.0981
F1 Score: 0.1784
Mean Reciprocal Rank (MRR): 0.9159
Mean Average Precision (MAP): 0.9159

Single-Match Analysis Results:
Success@10             : 98.12%
MRR: 0.9159
Mean Position: 1.29
Median Position: 1.0
Perfect Matches: 88.56%

Position Distribution:
------------------------------
Position_1: 88.56%
Position_2: 3.24%
Position_3: 1.49%
Position_4: 1.26%
Position_5: 1.38%
Position_6: 1.24%
Position_7: 0.30%
Position_8: 0.14%
Position_9: 0.28%
Position_10: 0.24%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.32%
Prefix_MRR: 0.9884
Prefix_Mean_Position: 1.02
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 98.64%

Prefix Position Distribution:
----------------------------------------
Position 1    : 98.64%
Position 2    : 0.15%
Position 3    : 0.20%
Position 4    : 0.10%
Position 5    : 0.08%
Position 6    : 0.00%
Position 7    : 0.07%
Position 8    : 0.04%
Position 9    : 0.01%
Position 1

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.9826
Precision: 0.0983
F1 Score: 0.1787
Mean Reciprocal Rank (MRR): 0.9183
Mean Average Precision (MAP): 0.9183

Single-Match Analysis Results:
Success@10             : 98.26%
MRR: 0.9183
Mean Position: 1.29
Median Position: 1.0
Perfect Matches: 88.86%

Position Distribution:
------------------------------
Position_1: 88.86%
Position_2: 3.21%
Position_3: 1.46%
Position_4: 1.01%
Position_5: 1.48%
Position_6: 1.15%
Position_7: 0.34%
Position_8: 0.25%
Position_9: 0.29%
Position_10: 0.20%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 99.41%
Prefix_MRR: 0.9868
Prefix_Mean_Position: 1.03
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 98.38%

Prefix Position Distribution:
----------------------------------------
Position 1    : 98.38%
Position 2    : 0.16%
Position 3    : 0.39%
Position 4    : 0.11%
Position 5    : 0.24%
Position 6    : 0.00%
Position 7    : 0.10%
Position 8    : 0.01%
Position 9    : 0.01%
Position 1

In [ ]:
k1= [0.05, 0.1, 0.15] 
b = [0.15, 0.20, 0.25]

matrix_single = [
    [97.81, 98.16, 98.10],  
    [97.90, 98.02,],  
    [,,]
]

matrix_prefix = [
    [99.33, 99.44, 99.31],  
    [99.39, 99.29,],  
    [,,]
]


from tabulate import tabulate

# Prepare header (with one empty cell top-left)
headers = ["k1 \\ b"] + columns

# Combine each row with its corresponding k1 value
table_single = [[rows[i]] + matrix_single[i] for i in range(len(rows))]
table_prefix = [[rows[i]] + matrix_prefix[i] for i in range(len(rows))]

# Print the table
print('Single-match')
print(tabulate(table_single, headers=headers, floatfmt=".2f"))
print('-'*50)
print('Prefix-match')
print(tabulate(table_prefix, headers=headers, floatfmt=".2f"))